<a href="https://colab.research.google.com/github/snumryk/TRPA1-ML-benchmark/blob/main/scripts/ChemBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Перевіримо що файли видно
import os
path = '/content/drive/MyDrive/trpa1_project'
print(os.listdir(path))

Mounted at /content/drive
['trpa1_antagonists.csv', 'decoys_clean.csv']


In [ ]:
!pip install transformers datasets accelerate -q
!pip install rdkit scikit-learn xgboost scipy -q

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 58.8 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128, CUDA: True, GPU: Tesla T4


## завантаження даних і підготовка

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, matthews_corrcoef
from scipy.stats import spearmanr

SEED = 42
THRESHOLD = 7.0
DATA_PATH = '/content/drive/MyDrive/trpa1_project'

df = pd.read_csv(f'{DATA_PATH}/trpa1_antagonists.csv')
print(f"Dataset: {len(df)} compounds")
print(f"Split: {df['split'].value_counts().to_dict()}")

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Dataset: 1645 compounds
Split: {'train': 1324, 'test': 161, 'val': 160}
Train: 1324, Val: 160, Test: 161


##  діагностика моделі

In [ ]:
"""
Check: load ChemBERTa as regression model (with proper head),
verify architecture before fine-tuning.
"""
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "DeepChem/ChemBERTa-77M-MTR"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# num_labels=1 + problem_type="regression" gives us a proper regression head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression",
    ignore_mismatched_sizes=True,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Quick test: one molecule through the model
test_input = tokenizer("O=C/C=C/c1ccccc1", return_tensors="pt")
with torch.no_grad():
    output = model(**test_input)
print(f"Output shape: {output.logits.shape}")  # should be (1, 1)
print(f"Predicted value: {output.logits.item():.4f}")
print("Model loaded and working.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.96k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                        | Status     | 
---------------------------+------------+-
norm_mean                  | UNEXPECTED | 
regression.out_proj.bias   | UNEXPECTED | 
regression.out_proj.weight | UNEXPECTED | 
regression.dense.bias      | UNEXPECTED | 
regression.dense.weight    | UNEXPECTED | 
norm_std                   | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 3,427,825
Trainable parameters: 3,427,825


model.safetensors:   0%|          | 0.00/14.0M [00:00<?, ?B/s]

Output shape: torch.Size([1, 1])
Predicted value: 0.0066
Model loaded and working.


## створення Dataset для HuggingFace Trainer

In [ ]:
from torch.utils.data import Dataset

class SMILESDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_length=128):
        self.smiles = smiles_list
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.smiles[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

train_dataset = SMILESDataset(train_df['std_smiles'].tolist(), train_df['pchembl_median'].values, tokenizer)
val_dataset   = SMILESDataset(val_df['std_smiles'].tolist(), val_df['pchembl_median'].values, tokenizer)
test_dataset  = SMILESDataset(test_df['std_smiles'].tolist(), test_df['pchembl_median'].values, tokenizer)

# Verify one sample
sample = train_dataset[0]
print(f"Input IDs shape: {sample['input_ids'].shape}")
print(f"Label: {sample['labels'].item():.2f}")
print("Datasets ready.")

Input IDs shape: torch.Size([128])
Label: 9.00
Datasets ready.


## Fine-tuning ChemBERTa (end-to-end)

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    r2 = r2_score(labels, predictions)
    rho = spearmanr(labels, predictions).correlation
    return {'rmse': rmse, 'r2': r2, 'spearman': rho}

training_args = TrainingArguments(
    output_dir='./chemberta_finetuned',
    num_train_epochs=50,               # upper bound, early stopping will cut
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,                # standard fine-tuning LR
    warmup_ratio=0.1,                  # 10% warmup
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,       # load best checkpoint after training
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=3,
    logging_steps=10,
    seed=SEED,
    fp16=True,                         # faster on T4
    report_to='none',                  # no wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print("Starting fine-tuning...")
train_result = trainer.train()
print(f"\nTraining complete. Best model loaded.")
print(f"Training loss: {train_result.training_loss:.4f}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Rmse,R2,Spearman
1,52.213904,48.665260,6.976049,-63.183098,0.352426
2,52.703467,47.835014,6.916286,-62.088108,0.515844
3,51.500421,45.833649,6.770055,-59.448570,0.535613
4,46.390414,40.120850,6.334102,-51.914127,0.543394
5,26.269879,19.788677,4.448447,-25.098669,0.552515
6,6.238058,4.014097,2.003521,-4.294067,0.468237
7,1.194740,1.322960,1.150200,-0.744811,0.502761
8,0.687725,1.011484,1.005725,-0.334014,0.532214
9,0.632694,0.886719,0.941658,-0.169466,0.557412
10,0.550270,0.798204,0.893423,-0.052727,0.569738


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete. Best model loaded.
Training loss: 7.8761


## оцінка на тест-сеті

In [ ]:
# Predict on test set
predictions_output = trainer.predict(test_dataset)
preds_ft = predictions_output.predictions.squeeze()
actuals = test_df['pchembl_median'].values

# Regression metrics
rmse_ft = np.sqrt(mean_squared_error(actuals, preds_ft))
r2_ft = r2_score(actuals, preds_ft)
rho_ft = spearmanr(actuals, preds_ft).correlation

# Classification metrics (threshold 7.0)
y_cls = (actuals >= THRESHOLD).astype(int)
auc_ft = roc_auc_score(y_cls, preds_ft)
mcc_ft = matthews_corrcoef(y_cls, (preds_ft >= THRESHOLD).astype(int))

# Bootstrap CI for R2
rng = np.random.default_rng(SEED)
boot_r2 = []
for _ in range(1000):
    idx = rng.integers(0, len(actuals), len(actuals))
    boot_r2.append(r2_score(actuals[idx], preds_ft[idx]))
r2_ci = np.percentile(boot_r2, [2.5, 97.5])

print("="*70)
print("ChemBERTa-2 FINE-TUNED (end-to-end) — Test Results")
print("="*70)
print(f"  RMSE:     {rmse_ft:.3f}")
print(f"  R2:       {r2_ft:.3f}   95% CI [{r2_ci[0]:.3f}, {r2_ci[1]:.3f}]")
print(f"  Spearman: {rho_ft:.3f}")
print(f"  AUC:      {auc_ft:.3f}")
print(f"  MCC:      {mcc_ft:.3f}")

ChemBERTa-2 FINE-TUNED (end-to-end) — Test Results
  RMSE:     0.787
  R2:       0.247   95% CI [0.021, 0.391]
  Spearman: 0.497
  AUC:      0.739
  MCC:      0.311


## frozen embeddings для порівняння

In [ ]:
"""
For direct comparison: frozen ChemBERTa embeddings → XGBoost.
Both CLS and Mean Pooling, same as local experiment.
"""
from transformers import AutoModel
from xgboost import XGBRegressor

# Reload base model (frozen, no regression head)
base_model = AutoModel.from_pretrained(MODEL_NAME).to('cuda').eval()

def extract_embeddings(smiles_list, pooling='cls'):
    embeddings = []
    for smi in smiles_list:
        tokens = tokenizer(smi, return_tensors="pt", truncation=True,
                           padding=True, max_length=128).to('cuda')
        with torch.no_grad():
            output = base_model(**tokens)
        hidden = output.last_hidden_state  # (1, seq_len, 384)
        if pooling == 'cls':
            emb = hidden[:, 0, :].cpu().numpy().ravel()
        else:  # mean pooling
            mask = tokens['attention_mask'].unsqueeze(-1).float()
            emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            emb = emb.cpu().numpy().ravel()
        embeddings.append(emb)
    return np.vstack(embeddings)

print("Extracting frozen embeddings (GPU-accelerated)...")
X_train_cls = extract_embeddings(train_df['std_smiles'].tolist(), 'cls')
X_test_cls  = extract_embeddings(test_df['std_smiles'].tolist(), 'cls')
print(f"CLS embeddings: {X_train_cls.shape}")

# RDKit descriptors
RDKIT_DESCS = [
    'MolWt', 'MolLogP', 'MolMR', 'TPSA',
    'NumHAcceptors', 'NumHDonors', 'NumRotatableBonds',
    'NumAromaticRings', 'RingCount', 'FractionCSP3',
    'HeavyAtomCount', 'NumAliphaticRings', 'NumSaturatedRings',
    'NumHeteroatoms', 'LabuteASA',
]

def compute_rdkit(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.zeros(len(RDKIT_DESCS))
    return np.array([float(getattr(Descriptors, n)(mol)) for n in RDKIT_DESCS], dtype=np.float32)

X_train_rdk = np.vstack(train_df['std_smiles'].apply(compute_rdkit).values)
X_test_rdk  = np.vstack(test_df['std_smiles'].apply(compute_rdkit).values)

# Combine CLS + RDKit
X_train_combo = np.hstack([X_train_cls, X_train_rdk])
X_test_combo  = np.hstack([X_test_cls, X_test_rdk])

y_train = train_df['pchembl_median'].values

# Run XGBoost on frozen embeddings
configs = {
    'XGB + CLS(384)':     (X_train_cls, X_test_cls),
    'XGB + CLS+RDKit':    (X_train_combo, X_test_combo),
}

print("\n" + "="*70)
print("FROZEN EMBEDDINGS + XGBoost (for comparison)")
print("="*70)

for name, (Xtr, Xte) in configs.items():
    xgb = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                       n_jobs=-1, random_state=SEED)
    xgb.fit(Xtr, y_train)
    preds = xgb.predict(Xte)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    r2 = r2_score(actuals, preds)
    rho = spearmanr(actuals, preds).correlation
    auc = roc_auc_score(y_cls, preds)
    print(f"  {name:<25} RMSE={rmse:.3f} R2={r2:.3f} Spearman={rho:.3f} AUC={auc:.3f}")

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                        | Status     | 
---------------------------+------------+-
norm_mean                  | UNEXPECTED | 
regression.out_proj.bias   | UNEXPECTED | 
regression.out_proj.weight | UNEXPECTED | 
regression.dense.bias      | UNEXPECTED | 
regression.dense.weight    | UNEXPECTED | 
norm_std                   | UNEXPECTED | 
pooler.dense.weight        | MISSING    | 
pooler.dense.bias          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Extracting frozen embeddings (GPU-accelerated)...
CLS embeddings: (1324, 384)

FROZEN EMBEDDINGS + XGBoost (for comparison)
  XGB + CLS(384)            RMSE=0.739 R2=0.337 Spearman=0.641 AUC=0.809
  XGB + CLS+RDKit           RMSE=0.705 R2=0.396 Spearman=0.662 AUC=0.806


## фінальна порівняльна таблиця

In [ ]:
print("\n" + "="*80)
print("COMPLETE COMPARISON: All experiments (local + Colab)")
print("="*80)
print(f"{'Model':<35} {'RMSE':>6} {'R2':>6} {'Spearman':>9} {'AUC':>6} {'Method'}")
print("-"*80)

rows = [
    ("RF (Morgan 2048)",           0.795, 0.232, 0.552, 0.795, "Fingerprint"),
    ("XGB (Morgan 2048)",          0.781, 0.257, 0.547, 0.790, "Fingerprint"),
    ("RF (15 RDKit)",              0.724, 0.363, 0.624, 0.800, "Descriptors"),
    ("D-MPNN (graph only)",        0.823, 0.176, 0.554, 0.761, "GNN"),
    ("D-MPNN+RDKit (ES)",          0.763, 0.292, 0.621, 0.810, "GNN+Desc"),
    ("XGB+CLS frozen (local)",     0.735, 0.344, 0.642, 0.806, "Frozen emb"),
    ("XGB+CLS+RDKit frozen (local)",0.706,0.395, 0.672, 0.811, "Frozen emb+Desc"),
]

for name, rmse, r2, rho, auc, method in rows:
    print(f"  {name:<35} {rmse:>6.3f} {r2:>6.3f} {rho:>9.3f} {auc:>6.3f}  {method}")

print()
print(f"  {'ChemBERTa FINE-TUNED':<35} {rmse_ft:>6.3f} {r2_ft:>6.3f} {rho_ft:>9.3f} {auc_ft:>6.3f}  Fine-tuned")

print("-"*80)
delta = r2_ft - 0.395
print(f"\nFine-tuned vs best frozen (XGB+CLS+RDKit): R2 delta = {delta:+.3f}")
delta2 = r2_ft - 0.363
print(f"Fine-tuned vs RF(15 RDKit):                 R2 delta = {delta2:+.3f}")

if r2_ft > 0.395:
    print("\n→ Fine-tuning IMPROVES over frozen embeddings.")
    print("  Write in paper: 'End-to-end fine-tuning of ChemBERTa-2 on TRPA1 data'")
    print("  'yielded further improvement over frozen feature extraction.'")
else:
    print("\n→ Fine-tuning does NOT improve over frozen embeddings.")
    print("  Write in paper: 'Frozen pretrained embeddings combined with")
    print("  'physicochemical descriptors proved sufficient; end-to-end fine-tuning")
    print("  'did not yield additional benefit, likely due to limited dataset size.'")
    print("  (This is itself an interesting finding for the small-data regime.)")


COMPLETE COMPARISON: All experiments (local + Colab)
Model                                 RMSE     R2  Spearman    AUC Method
--------------------------------------------------------------------------------
  RF (Morgan 2048)                     0.795  0.232     0.552  0.795  Fingerprint
  XGB (Morgan 2048)                    0.781  0.257     0.547  0.790  Fingerprint
  RF (15 RDKit)                        0.724  0.363     0.624  0.800  Descriptors
  D-MPNN (graph only)                  0.823  0.176     0.554  0.761  GNN
  D-MPNN+RDKit (ES)                    0.763  0.292     0.621  0.810  GNN+Desc
  XGB+CLS frozen (local)               0.735  0.344     0.642  0.806  Frozen emb
  XGB+CLS+RDKit frozen (local)         0.706  0.395     0.672  0.811  Frozen emb+Desc

  ChemBERTa FINE-TUNED                 0.787  0.247     0.497  0.739  Fine-tuned
--------------------------------------------------------------------------------

Fine-tuned vs best frozen (XGB+CLS+RDKit): R2 delta = -0.148
